# Transfer Gold Chunk และ Graph Extraction: Dry-run

Notebook นี้สาธิตลำดับจากการโหลด Gold/Distractor Chunk ไปจนถึงการเตรียมผล Graph Extraction โดยใช้ไฟล์ JSONL จริงของโปรเจกต์ แต่ **ไม่เชื่อมต่อหรือเขียนข้อมูลลง Neo4j และไม่เรียก LLM**

ส่วน DB ใช้ Recording Driver ที่บันทึกว่าโค้ดจะส่ง Cypher และข้อมูลอะไรไปบ้าง ส่วนผล Graph Extraction ใช้ตัวอย่างจำลองที่ระบุชัดเจนว่าไม่ใช่ผลจากโมเดล

เลือก Kernel ของ Python environment ที่ติดตั้ง dependencies ของโปรเจกต์ เช่น senior_project_py312 แล้วรันจากบนลงล่าง


## เส้นทางที่เราจะดู

1. อ่าน Gold และ Distractor JSONL
2. ตรวจตัวอย่าง Chunk และ embedding
3. เรียกฟังก์ชัน Transfer จริง แต่แทน Neo4j ด้วยตัวรับคำสั่งจำลอง
4. แสดง constraints และ index ที่ init_schema เตรียมไว้ โดยไม่รันคำสั่ง
5. สร้าง Chunk object ที่ Pilot_eval ใช้
6. จำลองผล extraction แล้วเรียกวงจร extract_chunks และ KGStore โดยใช้ตัวรับคำสั่งจำลอง

หมายเหตุ: Pilot_eval.load_chunks ปกติจะอ่าน Chunk จาก Neo4j ด้วยคำสั่งอ่าน ใน Notebook นี้ใช้ JSONL แทนเพื่อไม่ต้องเชื่อมต่อฐานข้อมูล


In [10]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "eval_scripts").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from eval_scripts import transfer_gold_chunks as transfer
from eval_scripts import Pilot_eval as pilot
from semigraph.offline.chunker import Chunk
from semigraph.offline.kg_store import KGStore, _CONSTRAINTS, _INDEXES
from semigraph.ontology.nodes import GraphExtractionResult, GraphNode, GraphRelationship

print("Project root:", ROOT)
print("Transfer input:", transfer.INPUT_FILE)
print("Database writes: disabled in this Notebook")


Project root: /home/kantinan/programming/project
Transfer input: /home/kantinan/programming/project/data/neo4j/finreflectkg_gold_chunks.jsonl
Database writes: disabled in this Notebook


## 1. อ่านไฟล์ JSONL

ฟังก์ชัน load_jsonl คืน list ของ dict โดยอ่านหนึ่ง JSON ต่อหนึ่งบรรทัด นี่คือข้อมูลที่ Transfer script รับเข้า


In [11]:
gold_chunks = transfer.load_jsonl(transfer.INPUT_FILE)
distractor_chunks = transfer.load_jsonl(transfer.DISTRACTOR_INPUT_FILE)

print("Gold records:", len(gold_chunks))
print("Distractor records:", len(distractor_chunks))
print("รวม records:", len(gold_chunks) + len(distractor_chunks))
print("ชนิด gold_chunks:", type(gold_chunks).__name__)
print("ชนิด record แรก:", type(gold_chunks[0]).__name__)
print("จำนวน fields:", len(gold_chunks[0]))
print("embedding dimensions:", len(gold_chunks[0].get("embedding", [])))
print("embedding model:", gold_chunks[0].get("embedding_model"))


Gold records: 82
Distractor records: 820
รวม records: 902
ชนิด gold_chunks: list
ชนิด record แรก: dict
จำนวน fields: 14
embedding dimensions: 768
embedding model: BAAI/bge-base-en-v1.5


## 2. ดูตัวอย่าง Chunk

ไม่พิมพ์ embedding ทั้ง 768 ค่า เพราะมีขนาดใหญ่ แสดงแค่ขนาดและข้อมูล provenance ที่ช่วยระบุที่มาของข้อความ


In [12]:
sample = gold_chunks[0]

for key in [
    "chunk_id", "ticker", "fiscal_year", "filing_type",
    "source_file", "page_id", "section", "char_count",
    "token_estimate", "embedding_model",
]:
    print(f"{key}: {sample.get(key)!r}")

print("embedding length:", len(sample.get("embedding", [])))
print("text preview:", sample.get("text", "")[:240])


chunk_id: 'AMD_10k_2022.pdf::page_5::chunk_1'
ticker: 'AMD'
fiscal_year: 2022
filing_type: '10-K'
source_file: 'AMD_10k_2022.pdf'
page_id: 'page_5'
section: 'page_5'
char_count: 3591
token_estimate: 897
embedding_model: 'BAAI/bge-base-en-v1.5'
embedding length: 768
text preview: ## Additional Information AMD was incorporated under the laws of Delaware on May 1, 1969 and became a publicly held company in 1972. Our common stock is currently listed on The NASDAQ Global Select Market (NASDAQ) under the symbol 'AMD'. Ou


## 3. เตรียมตัวรับ Cypher แบบจำลอง

Transfer script เรียก session.run(query, chunks=...) แต่คลาสด้านล่างเพียงบันทึกคำสั่งและสรุป parameter ไม่มี network connection และไม่มี Neo4j write


In [13]:
class DryRunResult:
    def consume(self):
        return None


class DryRunSession:
    def __init__(self, calls):
        self.calls = calls

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        return False

    def execute_write(self, callback):
        print("[dry-run] เรียก transaction callback ในหน่วยความจำ")
        return callback(self)

    def run(self, query, **params):
        call = {"query": query, "params": params}
        self.calls.append(call)
        flat_query = " ".join(query.split())
        print(f"[dry-run] Cypher: {flat_query[:145]}...")

        for key, value in params.items():
            if isinstance(value, list):
                print(f"           parameter {key}: {len(value)} records")
                if value and isinstance(value[0], dict):
                    first = value[0]
                    identity = first.get("chunk_id", first.get("name", first.get("id")))
                    print("           first record identifier:", identity)
                    print("           first record fields:", sorted(first.keys()))
                    if "embedding" in first:
                        print("           first record embedding dimensions:", len(first["embedding"]))
            else:
                print(f"           parameter {key}: {value!r}")

        return DryRunResult()


class DryRunDriver:
    def __init__(self):
        self.calls = []

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        return False

    def session(self):
        return DryRunSession(self.calls)


dry_driver = DryRunDriver()
print("สร้าง DryRunDriver แล้ว; ยังไม่มีการเปิด connection ไป Neo4j")


สร้าง DryRunDriver แล้ว; ยังไม่มีการเปิด connection ไป Neo4j


## 4. เรียก Transfer function โดยไม่เขียน DB

เราแทน controlled_driver ชั่วคราวด้วย DryRunDriver แล้วเรียกฟังก์ชันเดิมของ transfer_gold_chunks.py จริง


In [14]:
original_controlled_driver = transfer.controlled_driver
transfer.controlled_driver = lambda: dry_driver

try:
    transfer.write_chunks_to_neo4j(gold_chunks)
    loaded_distractors = transfer.write_distractor()
finally:
    transfer.controlled_driver = original_controlled_driver

print("Gold records ส่งเข้า dry-run:", len(gold_chunks))
print("Distractor records ส่งเข้า dry-run:", loaded_distractors)
print("จำนวน Cypher calls ที่บันทึก:", len(dry_driver.calls))


[dry-run] Cypher: UNWIND $chunks AS chunk MERGE (c:Chunk {chunk_id: chunk.chunk_id}) SET c += chunk...
           parameter chunks: 82 records
           first record identifier: AMD_10k_2022.pdf::page_5::chunk_1
           first record fields: ['char_count', 'chunk_id', 'dataset', 'embedding', 'embedding_model', 'filing_type', 'fiscal_year', 'page_id', 'raw_chunk_id', 'section', 'source_file', 'text', 'ticker', 'token_estimate']
           first record embedding dimensions: 768
[dry-run] Cypher: UNWIND $chunks AS chunk MERGE (c:Chunk {chunk_id: chunk.chunk_id}) SET c += chunk, c.is_distractor = true...
           parameter chunks: 820 records
           first record identifier: AMD_10k_2022.pdf::page_11::chunk_1
           first record fields: ['char_count', 'chunk_id', 'dataset', 'embedding', 'embedding_model', 'filing_type', 'fiscal_year', 'page_id', 'raw_chunk_id', 'section', 'source_file', 'text', 'ticker', 'token_estimate']
           first record embedding dimensions: 768
Gold r

### อ่านผลของ Transfer

- Gold ใช้ MERGE หา/สร้าง Chunk ด้วย chunk_id แล้ว SET c += chunk เพื่อคัดลอก properties จาก JSONL
- Distractor ใช้แนวทางเดียวกัน และเพิ่ม is_distractor = true
- จำนวนที่พิมพ์เป็นจำนวน records ที่ส่งให้ Cypher ไม่ได้แปลว่ามี node ใหม่เพิ่มเท่านั้น เพราะ MERGE อาจเจอ Chunk ID เดิม


## 5. ดูสิ่งที่ init_schema เตรียมไว้

Pilot_eval เรียก init_schema ก่อนเก็บผล extraction โค้ดด้านล่างแสดงคำสั่งที่จะใช้ แต่ไม่ได้ส่งคำสั่งไปฐานข้อมูล


In [15]:
schema_statements = _CONSTRAINTS + _INDEXES
print("จำนวน schema statements:", len(schema_statements))

for number, statement in enumerate(schema_statements, start=1):
    print(f"{number}. {' '.join(statement.split())}")

print("หมายเหตุ: รายการนี้ไม่มี CREATE VECTOR INDEX")


จำนวน schema statements: 5
1. CREATE CONSTRAINT entity_name_type IF NOT EXISTS FOR (e:Entity) REQUIRE (e.name, e.type) IS UNIQUE
2. CREATE CONSTRAINT chunk_id_unique IF NOT EXISTS FOR (c:Chunk) REQUIRE c.chunk_id IS UNIQUE
3. CREATE CONSTRAINT document_unique IF NOT EXISTS FOR (d:Document) REQUIRE (d.ticker, d.fiscal_year, d.filing_type) IS UNIQUE
4. CREATE CONSTRAINT section_unique IF NOT EXISTS FOR (s:Section) REQUIRE (s.doc_key, s.name) IS UNIQUE
5. CREATE INDEX entity_type IF NOT EXISTS FOR (e:Entity) ON (e.type)
หมายเหตุ: รายการนี้ไม่มี CREATE VECTOR INDEX


## 6. แปลง record เป็น Chunk model

Pilot_eval ปกติอ่าน properties จาก Neo4j แล้วสร้าง Chunk(**row) เราใช้ Gold JSONL record แทน และแปลง fiscal_year เป็น str ตามชนิดของ Chunk model


In [16]:
chunk = Chunk(
    chunk_id=sample["chunk_id"],
    ticker=sample["ticker"],
    fiscal_year=str(sample["fiscal_year"]),
    filing_type=sample["filing_type"],
    section=sample["section"],
    text=sample["text"],
    char_count=sample["char_count"],
    token_estimate=sample["token_estimate"],
)

print("ชนิดผลลัพธ์:", type(chunk).__name__)
print("Chunk metadata:", chunk.model_dump(exclude={"text"}))
print("ความยาว text:", len(chunk.text))
print("text preview:", chunk.text[:180])


ชนิดผลลัพธ์: Chunk
Chunk metadata: {'chunk_id': 'AMD_10k_2022.pdf::page_5::chunk_1', 'ticker': 'AMD', 'fiscal_year': '2022', 'filing_type': '10-K', 'section': 'page_5', 'char_count': 3591, 'token_estimate': 897}
ความยาว text: 3591
text preview: ## Additional Information AMD was incorporated under the laws of Delaware on May 1, 1969 and became a publicly held company in 1972. Our common stock is currently listed on The NAS


## 7. ตัวอย่างผล extraction

ผลด้านล่างเป็น **ข้อมูลจำลองเพื่อสาธิตรูปแบบ** ไม่ใช่ผลจาก LLM และไม่อ้างว่าเป็น facts จาก Chunk ตัวอย่างจริง เราจะใช้มันดูว่า Pilot_eval และ KGStore ส่งข้อมูลแบบไหนต่อ


In [17]:
mock_result = GraphExtractionResult(
    nodes=[
        GraphNode(id="sample company", type="ORG"),
        GraphNode(id="sample segment", type="SEGMENT"),
    ],
    relationships=[
        GraphRelationship(
            source="sample company",
            source_type="ORG",
            target="sample segment",
            target_type="SEGMENT",
            type="OPERATES_IN",
            properties={"fiscal_year": chunk.fiscal_year},
        )
    ],
)

print("nodes:", len(mock_result.nodes))
print("relationships:", len(mock_result.relationships))
print(mock_result.model_dump())


nodes: 2
relationships: 1
{'nodes': [{'id': 'sample company', 'type': 'ORG', 'properties': {'name': 'sample company'}}, {'id': 'sample segment', 'type': 'SEGMENT', 'properties': {'name': 'sample segment'}}], 'relationships': [{'source': 'sample company', 'source_type': 'ORG', 'target': 'sample segment', 'target_type': 'SEGMENT', 'type': 'OPERATES_IN', 'properties': {'fiscal_year': '2022'}}]}


## 8. จำลอง Pilot_eval.extract_chunks และ KGStore

เราจะเรียก extract_chunks ของ Pilot_eval จริง แต่แทน extract_chunk ด้วยฟังก์ชันที่คืน mock_result ข้างบน จากนั้น KGStore จริงจะสร้าง Cypher parameter payload แต่ DryRunDriver จะบันทึกแทนการเขียนฐานข้อมูล


In [18]:
original_extract_chunk = pilot.extract_chunk

def fake_extract_chunk(**kwargs):
    print("[mock extraction] received chunk_id:", kwargs["chunk_id"])
    print("[mock extraction] received text characters:", len(kwargs["text"]))
    print("[mock extraction] returned a prepared GraphExtractionResult")
    return mock_result

pilot.extract_chunk = fake_extract_chunk
store = KGStore(driver=dry_driver)

try:
    extracted = pilot.extract_chunks([chunk], llm=None, store=store)
finally:
    pilot.extract_chunk = original_extract_chunk

print("ผลลัพธ์ที่ Pilot_eval คืน:", len(extracted), "chunk-result pair")
print("ชนิดผล:", type(extracted[0][1]).__name__)
print("Cypher calls ทั้งหมดใน dry-run:", len(dry_driver.calls))


[mock extraction] received chunk_id: AMD_10k_2022.pdf::page_5::chunk_1
[mock extraction] received text characters: 3591
[mock extraction] returned a prepared GraphExtractionResult
[dry-run] เรียก transaction callback ในหน่วยความจำ
[dry-run] Cypher: MATCH ()-[r]->() WHERE r.source_chunk = $chunk_id DELETE r...
           parameter chunk_id: 'AMD_10k_2022.pdf::page_5::chunk_1'
[dry-run] Cypher: MATCH (:Chunk {chunk_id: $chunk_id})-[m:MENTIONS]->(:Entity) DELETE m...
           parameter chunk_id: 'AMD_10k_2022.pdf::page_5::chunk_1'
[dry-run] Cypher: MATCH (e:Entity) WHERE NOT (e)<-[:MENTIONS]-(:Chunk) DETACH DELETE e...
[dry-run] Cypher: MATCH (c:Chunk {chunk_id: $chunk_id}) UNWIND $nodes AS node MERGE (e:Entity {name: node.name, type: node.type}) ON CREATE SET e += node.propertie...
           parameter chunk_id: 'AMD_10k_2022.pdf::page_5::chunk_1'
           parameter nodes: 2 records
           first record identifier: sample company
           first record fields: ['name', 'propertie

## 9. ถ้าต้องการลอง LLM จริงภายหลัง

เซลล์นี้ปิดไว้โดยตั้งใจ เพราะจะส่งข้อความ Chunk ไปยัง LLM และอาจมีค่าใช้จ่าย เมื่อรันแล้วก็ยังไม่เขียน Neo4j เพราะเรียก extract_chunk โดยไม่มี KGStore


In [19]:
# เปิดใช้เองเมื่อพร้อมเรียก LLM:
#
# from semigraph.connections import get_llm
# cfg = pilot.controlled_config()
# llm = get_llm(cfg)
# real_result = pilot.extract_chunk(
#     text=chunk.text,
#     section=pilot.FULL_ONTOLOGY,
#     llm=llm,
#     chunk_id=chunk.chunk_id,
#     filer_ticker=chunk.ticker,
# )
# print(real_result.model_dump())


## สรุป

- Transfer script รับ JSONL แล้วส่ง Chunk properties ไปยัง Neo4j; DryRunDriver ทำให้เราเห็นคำสั่งและข้อมูลโดยไม่เขียน DB
- init_schema สร้าง unique constraints และ index บน Entity.type; ไม่ได้สร้าง vector index
- Pilot_eval รับ Chunk text แล้วเรียก extraction; เมื่อมี KGStore จึงเก็บ Entity, MENTIONS และความสัมพันธ์
- ใน Notebook นี้ผล extraction เป็น mock; หากต้องการผลจากโมเดลจริง ให้เปิดเซลล์ LLM ด้วยตัวเอง
- ตัวอย่างนี้จึงสาธิตโครงสร้างและข้อมูลที่ไหลผ่านแต่ละขั้น โดยไม่มี DB write และไม่มี LLM call
